<a href="https://colab.research.google.com/github/kaykesm/MLCB_CK_2026/blob/main/MLCB_CK_2026/AULA_07%20/lab01_aula07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =====================================================================
# LAB 1: NLU BANCÁRIO COM REGRESSÃO LOGÍSTICA
# =====================================================================

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

dados_treino = [
    ("Quero fazer uma transferencia de 500 reais", "transferencia"),
    ("Preciso transferir dinheiro para outra conta", "transferencia"),
    ("Quero consultar o saldo da minha conta", "consultar_saldo"),
    ("Quanto dinheiro tenho na conta", "consultar_saldo"),
    ("Quero pagar um boleto de 200 reais", "pagar_boleto"),
    ("Preciso realizar o pagamento de um boleto", "pagar_boleto"),
    ("Qual o horario de funcionamento da agencia", "fora_escopo")
]

stopwords = ["o", "a", "e", "de", "da", "do", "para", "um", "uma"]

# 1. PRÉ-PROCESSAMENTO
def pré_processar(texto):
    texto_limpo = re.sub(r'[^\w\s]', '', texto.lower()).strip()
    tokens = texto_limpo.split()

    # TODO 1.1
    tokens_filtrados = [token for token in tokens if token not in stopwords]

    return " ".join(tokens_filtrados)

X_treino_limpo = [pré_processar(item[0]) for item in dados_treino]
y_treino = [item[1] for item in dados_treino]

# 2. VETORIZAÇÃO E TREINAMENTO
vectorizer = TfidfVectorizer()

# TODO 1.2
X_vetorizado = vectorizer.fit_transform(X_treino_limpo)

modelo = LogisticRegression()
modelo.fit(X_vetorizado, y_treino)

# 3. EXTRAÇÃO DE ENTIDADE (REGEX)
def extrair_valor_reais(texto_original):
    match = re.search(r'(\d+)\s*reais', texto_original, re.IGNORECASE)

    # TODO 1.3
    if match:
        return float(match.group(1))
    else:
        return None

# 4. PIPELINE NLU
def processar_nlu(mensagem_usuario, threshold=0.55):
    texto_p = pré_processar(mensagem_usuario)
    vetor_input = vectorizer.transform([texto_p])

    probas = modelo.predict_proba(vetor_input)[0]

    # TODO 1.4
    maior_confianca = max(probas)
    intencao_prevista = modelo.classes_[probas.argmax()]

    # TODO 1.5
    if maior_confianca < threshold or intencao_prevista == "fora_escopo":
        return {
            "status": "FALLBACK"
        }
    else:
        return {
            "status": "SUCESSO",
            "intencao": intencao_prevista,
            "confianca": maior_confianca,
            "valor": extrair_valor_reais(mensagem_usuario)
        }

# --- TESTE 1 ---
if __name__ == "__main__":
    print(processar_nlu("Quero fazer uma transferencia de 500 reais"))

{'status': 'FALLBACK'}


In [2]:
print(extrair_valor_reais("Quero fazer uma transferencia de 500 reais"))

500.0
